# Downstream CMAPSS_RUL - 3 modos (from_scratch + linear_probing + full_finetuning_lr1e-5)

Notebook de ejecucion del bloque downstream regresion RUL sobre el
dataset reconstruido por el builder dedicado (no por la harmonization
v0.5; ver `results/downstream/cmapss_rul_decision/decision.md`).

## Que hace

3 corridas en Colab Pro+ A100-SXM4-80GB:

1. `from_scratch`: backbone PatchTSTPhm random + cabeza random (baseline).
2. `linear_probing`: backbone SSL central full congelado + cabeza entrenable.
3. `full_finetuning_lr1e-5`: backbone SSL central full + cabeza, ambos entrenables.

Comparacion: la hipotesis principal del TFM es que el SSL central
transfiere a CMAPSS_RUL. Si `linear_probing` o `full_finetuning_lr1e-5`
tienen RMSE menor que `from_scratch`, la hipotesis se confirma en RUL.

## Que NO hace

- **NO ejecuta entrenamientos reales sin autorizacion explicita**: las
  celdas de entrenamiento traen un aviso claro y el usuario debe
  ejecutarlas a mano una por una tras confirmar GPU=A100.
- **NO usa la harmonization v0.5 de CMAPSS** (`processed/CMAPSS/`).
  El target ahi es ambiguo entre splits. Usa el dataset reconstruido
  por el builder en `processed_downstream/CMAPSS_RUL/`.
- **NO mezcla con el bloque federado**. Esto es central vs from_scratch
  sobre CMAPSS_RUL.
- **NO modifica los 3 YAML existentes** (commit `baef887`, blindados por
  test `tests/test_downstream_cmapss_rul_configs.py` commit `8696bc2`).

## Restricciones operativas

- **GPU A100 obligatoria**. CMAPSS_RUL tiene C=24 canales y B=21
  adaptativo (B*C=504, justo bajo el cap 512); ~11.8k samples train,
  20 epochs. Sin A100 el coste de cada corrida sube.
- Si el dry-run de alguna celda 9.x falla, parar y diagnosticar.
- Las 3 corridas se hacen **una a una** (no en paralelo), ~50 min
  cada una en A100. Total ~2.5 h.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. colab_init

In [ ]:
!bash /content/drive/MyDrive/fm_fl_phmd/colab_init.sh

## 3. Pull al HEAD

Debe incluir los 3 configs CMAPSS_RUL (commit `baef887`), el test
preflight `tests/test_downstream_cmapss_rul_configs.py` (commit
`8696bc2`) y este notebook.

In [ ]:
%cd /content/fm_fl_phmd
!git pull --ff-only
!git log -5 --oneline
!git status --porcelain

## 4. Verificar GPU

**Si NO es A100, parar aqui** y vuelve con sesion A100.

In [ ]:
!nvidia-smi | head -15

## 5. pytest preflight

Cuatro grupos de tests relevantes:
- `test_downstream_cmapss_rul_configs.py` (blindaje de los 3 YAML).
- `test_train_downstream_rul.py` (trainer RUL end-to-end sintetico).
- `test_downstream_regression_head.py` (RegressionHead).
- `test_downstream_regression_metrics.py` (MAE/RMSE/R2/CMAPSS score).

Si esto falla, **no continuar**.

In [ ]:
!python -m pytest \
  tests/test_downstream_cmapss_rul_configs.py \
  tests/test_train_downstream_rul.py \
  tests/test_downstream_regression_head.py \
  tests/test_downstream_regression_metrics.py -q

## 6. Verificar el dataset CMAPSS_RUL reconstruido

El builder dedicado escribio el dataset en `processed_downstream/CMAPSS_RUL/`.
Debe haber 23 shards: 12 train + 4 val + 7 test.

Tambien verificamos que el ckpt SSL central full existe.

In [ ]:
import json
from pathlib import Path

RUL_ROOT = Path("/content/drive/MyDrive/fm_fl_phmd/processed_downstream/CMAPSS_RUL")
SSL_CENTRAL_CKPT = Path(
    "/content/drive/MyDrive/fm_fl_phmd/checkpoints/"
    "ssl_central_full/ssl_central_full_patchtst_phm/ckpt_step100000.pt"
)

assert RUL_ROOT.is_dir(), f"NO existe {RUL_ROOT}; ejecutar antes el builder CMAPSS_RUL."
assert (RUL_ROOT / "manifest.json").is_file(), "falta manifest.json"
assert (RUL_ROOT / "done.flag").is_file(), "falta done.flag"

manifest = json.loads((RUL_ROOT / "manifest.json").read_text(encoding='utf-8'))
print("=== manifest CMAPSS_RUL ===")
for k in (
    "pipeline_config_hash", "pipeline_code_version",
    "target_key_default", "n_channels", "n_units_total",
):
    if k in manifest:
        print(f"  {k}: {manifest[k]}")

for split in ("train", "val", "test"):
    sd = manifest.get("splits", {}).get(split, {})
    n_shards = sd.get("n_shards")
    n_samples = sd.get("n_samples")
    print(f"  {split}: n_shards={n_shards}, n_samples={n_samples}")

assert SSL_CENTRAL_CKPT.is_file(), (
    f"NO existe ckpt SSL central: {SSL_CENTRAL_CKPT}"
)
!ls -lh "{SSL_CENTRAL_CKPT}"

# Listado de shards.
for split in ("train", "val", "test"):
    files = sorted((RUL_ROOT / split).glob("*.tar"))
    print(f"  {split}: {len(files)} shards (primer = {files[0].name if files else 'NONE'})")

## 7. Preparar `_stdout/`

In [ ]:
!mkdir -p /content/drive/MyDrive/fm_fl_phmd/logs/downstream/cmapss_rul/_stdout

## 8. Variable comun: ckpt SSL central

`CKPT_SC_STR` se define una sola vez para los 2 dry-runs SSL (linear
+ full) y las 2 corridas SSL reales. `from_scratch` no necesita
checkpoint.

In [ ]:
CKPT_SC_STR = (
    "/content/drive/MyDrive/fm_fl_phmd/checkpoints/"
    "ssl_central_full/ssl_central_full_patchtst_phm/"
    "ckpt_step100000.pt"
)
print('CKPT_SC_STR =', CKPT_SC_STR)

## 9. Dry-runs de los 3 configs (NO entrenan)

Para cada modo: lee manifest CMAPSS_RUL, valida shapes, carga ckpt
SSL si procede, construye modelo. Si algo no cuadra, parar antes
de entrenar.

### 9.1 Dry-run from_scratch (sin checkpoint)

In [ ]:
!python -m training.train_downstream_rul \
  --config training/configs/downstream_cmapss_rul_from_scratch.yaml \
  --mode from_scratch \
  --dry-run 2>&1 | tail -25

### 9.2 Dry-run linear_probing (con ckpt SSL central)

In [ ]:
!python -m training.train_downstream_rul \
  --config training/configs/downstream_cmapss_rul_linear_probing.yaml \
  --mode linear_probing \
  --checkpoint "{CKPT_SC_STR}" \
  --dry-run 2>&1 | tail -25

### 9.3 Dry-run full_finetuning_lr1e-5 (con ckpt SSL central)

In [ ]:
!python -m training.train_downstream_rul \
  --config training/configs/downstream_cmapss_rul_full_finetuning_lr1e-5.yaml \
  --mode full_finetuning \
  --checkpoint "{CKPT_SC_STR}" \
  --dry-run 2>&1 | tail -25

---

# AVISO IMPORTANTE

Las **3 celdas de entrenamiento** siguientes NO se ejecutan
automaticamente. Cada una entrena ~50 min en A100; total ~2.5 h.

**Antes de ejecutar cualquiera de las 3 celdas siguientes**:

- confirmar `nvidia-smi` = A100;
- confirmar que el pytest de la celda 5 paso;
- confirmar que la celda 6 imprimio el manifest (23 shards) y el ckpt SSL central;
- confirmar que los 3 dry-runs de la celda 9 imprimieron `n_channels=24`,
  `effective_bc<=512` (esperado 504 con B=21*C=24), forward sintetico OK.

Las corridas se hacen **una a una** (no en paralelo). El nombre del
stdout va a Drive con el `run_name` exacto para trazabilidad.

## 10. Corrida 1/3 - from_scratch (~50 min A100)

Baseline: backbone PatchTSTPhm random + cabeza random. `lr_head=1e-3`,
`lr_backbone=1e-4`.

In [ ]:
import time
t0 = time.time()
!python -m training.train_downstream_rul \
  --config training/configs/downstream_cmapss_rul_from_scratch.yaml \
  --mode from_scratch \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/downstream/cmapss_rul/_stdout/downstream_cmapss_rul_from_scratch.stdout.log
print(f"\n[total] CMAPSS_RUL from_scratch elapsed = {time.time() - t0:.1f} s")

## 11. Corrida 2/3 - linear_probing (~50 min A100)

Backbone SSL central congelado + cabeza entrenable. `lr_head=1e-3`,
`lr_backbone=null` (backbone congelado por `--mode linear_probing`).

In [ ]:
import time
t0 = time.time()
!python -m training.train_downstream_rul \
  --config training/configs/downstream_cmapss_rul_linear_probing.yaml \
  --mode linear_probing \
  --checkpoint "{CKPT_SC_STR}" \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/downstream/cmapss_rul/_stdout/downstream_cmapss_rul_linear_probing.stdout.log
print(f"\n[total] CMAPSS_RUL linear_probing elapsed = {time.time() - t0:.1f} s")

## 12. Corrida 3/3 - full_finetuning lr_backbone=1e-5 (~50 min A100)

Backbone SSL central entrenable junto con la cabeza. `lr_head=1e-3`,
`lr_backbone=1e-5` (sweet spot validado en CWRU + HSG18; 1e-4 produjo
catastrophic forgetting alli).

In [ ]:
import time
t0 = time.time()
!python -m training.train_downstream_rul \
  --config training/configs/downstream_cmapss_rul_full_finetuning_lr1e-5.yaml \
  --mode full_finetuning \
  --checkpoint "{CKPT_SC_STR}" \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/downstream/cmapss_rul/_stdout/downstream_cmapss_rul_full_finetuning_lr1e-5.stdout.log
print(f"\n[total] CMAPSS_RUL full_finetuning lr1e-5 elapsed = {time.time() - t0:.1f} s")

## 13. Summary: 3 modos lado a lado

Lee los 3 `run_info.json` generados por las celdas 10-12 y construye
una tabla con MAE, RMSE, R2 y CMAPSS score por modo.

El criterio formal para `metric_for_best` es `rmse_val` con
`lower_is_better=true`. La hipotesis principal del TFM ("SSL transfiere")
se confirma en RUL si el RMSE de `linear_probing` o `full_finetuning`
es **menor** que el de `from_scratch`.

In [ ]:
import json
from pathlib import Path

base = Path("/content/drive/MyDrive/fm_fl_phmd/logs/downstream/cmapss_rul")
runs = {
    "from_scratch":             base / "downstream_cmapss_rul_from_scratch"             / "run_info.json",
    "linear_probing":           base / "downstream_cmapss_rul_linear_probing"           / "run_info.json",
    "full_finetuning_lr1e-5":   base / "downstream_cmapss_rul_full_finetuning_lr1e-5"   / "run_info.json",
}

rows = {}
for mode, p in runs.items():
    if not p.is_file():
        print(f"WARN: no encontrado {p}; saltando")
        continue
    ri = json.loads(p.read_text(encoding="utf-8"))
    tm = ri.get("test_metrics") or {}
    rows[mode] = {
        "mae":             tm.get("mae"),
        "rmse":            tm.get("rmse"),
        "r2":              tm.get("r2"),
        "cmapss_score":    tm.get("cmapss_score", tm.get("score")),
        "n_samples":       tm.get("n_samples"),
        "best_epoch":      ri.get("best_epoch"),
        "best_value_val":  ri.get("best_value"),
        "elapsed_seconds": ri.get("elapsed_seconds"),
        "amp_nonfinite_grad_steps": ri.get("amp_nonfinite_grad_steps"),
        "config_hash":     ri.get("config_hash"),
        "git_hash":        ri.get("git_hash"),
        "checkpoint_loaded": ri.get("checkpoint_loaded"),
        "lr_head":         (ri.get("training") or {}).get("lr_head"),
        "lr_backbone":     (ri.get("training") or {}).get("lr_backbone"),
    }

def _f(x, fmt='.4f'):
    return ('n/a' if x is None else format(float(x), fmt))

print("=== CMAPSS_RUL downstream (3 corridas) ===")
print(f"{'mode':<24s} {'MAE':>10s} {'RMSE':>10s} {'R2':>10s} {'score':>12s} {'best_ep':>8s} {'best_val':>10s} {'elapsed':>10s} {'amp_nf':>7s} {'lr_bb':>10s}")
print('-' * 130)
for mode, r in rows.items():
    lr_bb = r.get('lr_backbone')
    lr_bb_str = 'null' if lr_bb is None else f"{float(lr_bb):.0e}"
    print(
        f"{mode:<24s} {_f(r['mae']):>10s} {_f(r['rmse']):>10s} "
        f"{_f(r['r2']):>10s} {_f(r['cmapss_score'], '.2f'):>12s} "
        f"{(r['best_epoch'] if r['best_epoch'] is not None else 'n/a'):>8} "
        f"{_f(r['best_value_val']):>10s} "
        f"{(f'{r["elapsed_seconds"]:.1f}s' if r['elapsed_seconds'] is not None else 'n/a'):>10s} "
        f"{(str(r['amp_nonfinite_grad_steps']) if r['amp_nonfinite_grad_steps'] is not None else 'n/a'):>7s} "
        f"{lr_bb_str:>10s}"
    )

# Verificacion: ckpt cargado en los modos SSL.
SC_PATH = (
    "/content/drive/MyDrive/fm_fl_phmd/checkpoints/"
    "ssl_central_full/ssl_central_full_patchtst_phm/ckpt_step100000.pt"
)
for mode in ("linear_probing", "full_finetuning_lr1e-5"):
    if mode in rows:
        ckpt = str(rows[mode]["checkpoint_loaded"])
        assert SC_PATH in ckpt, (
            f"{mode}: ckpt_loaded NO contiene el ckpt SSL central:\n{ckpt!r}"
        )
if "from_scratch" in rows:
    ckpt = rows["from_scratch"]["checkpoint_loaded"]
    assert ckpt is None or ckpt == "" or "ssl_central" not in str(ckpt), (
        f"from_scratch NO debe cargar ckpt SSL: {ckpt!r}"
    )
print("\nVerificado: linear_probing y full_finetuning usaron el ckpt SSL central; "
      "from_scratch entreno desde init random.")

## 14. Deltas: SSL transfiere a CMAPSS_RUL?

Comparacion de las 3 metricas test entre SSL (linear / full) y
`from_scratch`. Para RMSE/MAE/CMAPSS score, **menor es mejor**; para R2,
**mayor es mejor**.

Hipotesis principal del TFM: el SSL central transfiere a CMAPSS_RUL si
alguno de los modos SSL (linear o full) produce **menor RMSE** que
`from_scratch`.

In [ ]:
if 'from_scratch' in rows and rows['from_scratch'].get('rmse') is not None:
    fs = rows['from_scratch']
    print('=== Deltas SSL vs from_scratch (menor RMSE/MAE/score = mejor; mayor R2 = mejor) ===')
    print(f"{'mode':<24s} {'d_MAE':>10s} {'d_RMSE':>10s} {'d_R2':>10s} {'d_score':>12s}")
    print('-' * 70)
    print(f"{'from_scratch (ref)':<24s} {0:>10.4f} {0:>10.4f} {0:>10.4f} {0:>12.2f}")
    for mode in ('linear_probing', 'full_finetuning_lr1e-5'):
        if mode not in rows:
            print(f"{mode:<24s} {'(missing)':>10s}")
            continue
        r = rows[mode]
        d_mae = (r['mae'] - fs['mae']) if (r.get('mae') is not None and fs.get('mae') is not None) else None
        d_rmse = (r['rmse'] - fs['rmse']) if (r.get('rmse') is not None and fs.get('rmse') is not None) else None
        d_r2 = (r['r2'] - fs['r2']) if (r.get('r2') is not None and fs.get('r2') is not None) else None
        d_score = (r['cmapss_score'] - fs['cmapss_score']) if (r.get('cmapss_score') is not None and fs.get('cmapss_score') is not None) else None
        print(
            f"{mode:<24s} {_f(d_mae, '+10.4f'):>10s} {_f(d_rmse, '+10.4f'):>10s} "
            f"{_f(d_r2, '+10.4f'):>10s} {_f(d_score, '+12.2f'):>12s}"
        )

    print('\nLectura rapida:')
    print('  - d_RMSE < 0 (negativo) -> SSL mejora vs baseline; tanto mas negativo, mejor.')
    print('  - d_R2  > 0 (positivo) -> SSL mejora vs baseline.')
    print('  - d_score < 0 (negativo) -> SSL mejora vs baseline en metrica CMAPSS oficial.')
else:
    print('Sin from_scratch ejecutado todavia; faltan deltas. Ejecutar celdas 10-12 antes.')

## 15. Propuesta automatica de verdict (informativa)

Criterios formales:

- **CONFIRMADA** la hipotesis principal del TFM en RUL si
  `min(rmse_linear, rmse_full) < rmse_from_scratch`.
- **DEBIL** si la mejora es < 1% relativo del baseline.
- **NO CONFIRMADA** si ambos modos SSL tienen RMSE >= from_scratch.

La propuesta es **informativa** y depende solo del run real; no
autoriza ninguna accion automatica. La interpretacion final la hace
el usuario tras leer toda la evidencia.

In [ ]:
def _verdict_ssl_transfiere(rows):
    if 'from_scratch' not in rows or rows['from_scratch'].get('rmse') is None:
        return ('PENDIENTE', 'Falta from_scratch ejecutado.')
    fs_rmse = float(rows['from_scratch']['rmse'])
    best_ssl_rmse = None
    best_mode = None
    for mode in ('linear_probing', 'full_finetuning_lr1e-5'):
        if mode in rows and rows[mode].get('rmse') is not None:
            r = float(rows[mode]['rmse'])
            if best_ssl_rmse is None or r < best_ssl_rmse:
                best_ssl_rmse = r
                best_mode = mode
    if best_ssl_rmse is None:
        return ('PENDIENTE', 'Faltan modos SSL ejecutados.')
    delta = best_ssl_rmse - fs_rmse
    rel = delta / max(1e-9, abs(fs_rmse))
    if delta < 0:
        if rel <= -0.01:
            return ('CONFIRMADA', f'mejor modo SSL = {best_mode} con RMSE={best_ssl_rmse:.4f} vs from_scratch={fs_rmse:.4f} (delta={delta:+.4f}, {rel*100:+.2f} %).')
        return ('DEBIL', f'mejor modo SSL = {best_mode} mejora pero <1 % relativo (delta={delta:+.4f}, {rel*100:+.2f} %).')
    return ('NO_CONFIRMADA', f'ningun modo SSL mejora; mejor = {best_mode} con RMSE={best_ssl_rmse:.4f} vs from_scratch={fs_rmse:.4f} (delta={delta:+.4f}, {rel*100:+.2f} %).')

if rows:
    verdict, motivo = _verdict_ssl_transfiere(rows)
    print(f"=== Propuesta automatica (informativa) ===")
    print(f"verdict: {verdict}")
    print(f"motivo : {motivo}")
    print('\nRecordatorio: esta propuesta NO autoriza ninguna accion.')
    print('La interpretacion final la hace el usuario tras leer toda la evidencia y')
    print('versionar los 3 run_info.json en results/downstream/cmapss_rul/.')
else:
    print('Sin rows aun: ejecutar las celdas 10-12 antes de evaluar el verdict.')

## 16. Cierre

Pega el output completo de las celdas 13-15 en el chat para cerrar
el bloque downstream CMAPSS_RUL.

Lo que ocurrira despues, con autorizacion explicita:

1. Versionar los 3 `run_info.json` reales bit-a-bit en
   `results/downstream/cmapss_rul/{from_scratch,linear_probing,full_finetuning_lr1e-5}/run_info.json`,
   mismo patron que los bloques classification.
2. Crear `results/downstream/cmapss_rul/summary_rul_cmapss.json` con
   los numeros reales, deltas vs from_scratch, y verdict.
3. Crear `results/downstream/cmapss_rul/README.md` con lectura honesta,
   tablas y comparativa contra el bloque RUL preparatorio
   (`training/analyze_cmapss_rul_semantics.py` + decision.md).
4. Actualizar la tabla de "hipotesis principal del TFM" en
   `results/downstream/fl_pilot_vs_central/README.md` y
   `results/downstream/fl_fedprox_pilot_vs_central/README.md`
   con el resultado RUL ya confirmado o refutado (no especulado).

Hasta entonces: **no inventar metricas RUL**. Las dos tablas que
hablan de RUL en el repo actualmente dicen "CMAPSS RUL pendiente /
fuera de este bloque" (commit `212c178`); deben actualizarse solo con
los datos reales tras ejecutar este notebook.